# Book Recommendation Engine using KNN

This notebook builds a **book recommendation system** using the K-Nearest Neighbors (KNN) algorithm. Given a book title, it finds 5 similar books based on user rating patterns.

The idea: if two books are rated similarly by the same users, they're probably similar in genre or appeal — so recommending one to a reader who liked the other makes sense.


In [1]:
# import libraries
import os
import pandas as pd
from sklearn.neighbors import NearestNeighbors

## 1. Download & load the data

The **Book-Crossings dataset** contains 1.1 million ratings (scale 1–10) of 270,000 books by 90,000 users. It comes in two CSV files:

- `BX-Books.csv` — book metadata (ISBN, title, author)
- `BX-Book-Ratings.csv` — user ratings (user ID, ISBN, rating)


In [2]:
# get data files (skip if already present)
if not os.path.exists("BX-Books.csv"):
    !wget -q https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
    !unzip -q book-crossings.zip

books_filename = "BX-Books.csv"
ratings_filename = "BX-Book-Ratings.csv"

In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding="ISO-8859-1",
    sep=";",
    header=0,
    names=["isbn", "title", "author"],
    usecols=["isbn", "title", "author"],
    dtype={"isbn": "str", "title": "str", "author": "str"},
)

df_ratings = pd.read_csv(
    ratings_filename,
    encoding="ISO-8859-1",
    sep=";",
    header=0,
    names=["user", "isbn", "rating"],
    usecols=["user", "isbn", "rating"],
    dtype={"user": "int32", "isbn": "str", "rating": "float32"},
)

## 2. Inspect the data

Let's look at the structure of both DataFrames to understand what we're working with.


In [4]:
df_books.head()

,isbn,title,author
0,0195153448,Classical Mythology,Mark P. O. Morford
1,0002005018,Clara Callan,Richard Bruce Wright
2,0060973129,Decision in Normandy,Carlo D'Este
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata
4,0393045218,The Mummies of Urumchi,E. J. W. Barber


In [5]:
df_books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271379 entries, 0 to 271378
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   isbn    271379 non-null  object
 1   title   271379 non-null  object
 2   author  271377 non-null  object
dtypes: object(3)
memory usage: 6.2+ MB


In [6]:
df_ratings.head()

,user,isbn,rating
0,276725,034545104X,0.0
1,276726,0155061224,5.0
2,276727,0446520802,0.0
3,276729,052165615X,3.0
4,276729,0521795028,6.0


In [7]:
df_ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1149780 entries, 0 to 1149779
Data columns (total 3 columns):
 #   Column  Non-Null Count    Dtype  
---  ------  --------------    -----  
 0   user    1149780 non-null  int32  
 1   isbn    1149780 non-null  object 
 2   rating  1149780 non-null  float32
dtypes: float32(1), int32(1), object(1)
memory usage: 17.5+ MB


## 3. Filter for statistical significance

Most books are rarely rated. To get meaningful recommendations, we remove:

- **Users** with fewer than 200 ratings (they don't have enough taste data)
- **Books** with fewer than 100 ratings (not enough consensus on them)

First, we count how many ratings each user and each book has:


In [8]:
# find the frequency of user
count_user = df_ratings["user"].value_counts()
count_user

,count
user,
11676,13602
198711,7550
153662,6109
98391,5891
35859,5850
...,...
119573,1
276706,1
276697,1


In [9]:
# find the frequency of book
count_book = df_ratings["isbn"].value_counts()
count_book

,count
isbn,
0971880107,2502
0316666343,1295
0385504209,883
0060928336,732
0312195516,723
...,...
0671883917,1
0743257502,1
0767409752,1


In [10]:
# filter data — keep only users with >= 200 ratings and books with >= 100 ratings
df_ratings = df_ratings[~df_ratings["user"].isin(count_user[count_user < 200].index)]
df_ratings = df_ratings[~df_ratings["isbn"].isin(count_book[count_book < 100].index)]

## 4. Merge & create a pivot table

We merge the books and ratings into one DataFrame, then create a **pivot table** where:

- Each **row** = a book title
- Each **column** = a user
- Each **cell** = that user's rating for that book (0 if not rated)

This matrix is what KNN uses to find similar books.


In [11]:
# merge both dataframes
books_ratings = pd.merge(left=df_books, right=df_ratings, on="isbn")
books_ratings.head()

,isbn,title,author,user,rating
0,0440234743,The Testament,John Grisham,277478,0.0
1,0440234743,The Testament,John Grisham,2977,0.0
2,0440234743,The Testament,John Grisham,3363,0.0
3,0440234743,The Testament,John Grisham,7346,9.0
4,0440234743,The Testament,John Grisham,9856,0.0


In [12]:
# drop duplicates (same user rating same book title twice)
books_ratings = books_ratings.drop_duplicates(subset=["title", "user"])

# make pivot table for knn
pivot = books_ratings.pivot(index="title", columns="user", values="rating").fillna(0)
pivot

user,254,2276,2766,2977,3363,4017,4385,6242,6251,6323,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Beautiful Mind: The Life of Mathematical Genius and Nobel Laureate John Nash,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Without Remorse,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Year of Wonders,0.0,0.0,0.0,7.0,0.0,0.0,0.0,7.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5. Build the KNN model

We use `NearestNeighbors` from scikit-learn with **cosine distance** — it measures the angle between two rating vectors, ignoring their magnitude. Books that are rated similarly by the same users will have a small cosine distance.


In [13]:
# knn model
knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(pivot)

NearestNeighbors(algorithm='brute', metric='cosine')

## 6. The `get_recommends()` function

Give it a book title → it returns `[book_title, [[rec1, dist1], [rec2, dist2], ...]]`.

The function:

1. Looks up the book's rating vector from the pivot table
2. Finds the 6 nearest neighbors (including itself)
3. Skips the book itself
4. Returns the 5 nearest, sorted farthest → closest (as the challenge expects)


In [14]:
# function to return recommended books - this will be tested
def get_recommends(book=""):
    recommended_books = [book]

    try:
        x = pivot.loc[[book]].values.reshape(1, -1)
    except KeyError:
        return [book, []]

    distances, indexes = knn.kneighbors(x, n_neighbors=6)

    list_recommended_books = []
    for distance, index in zip(distances[0], indexes[0]):
        if pivot.index[index] != book:
            list_recommended_books.append([pivot.index[index], distance])

    recommended_books.append(list_recommended_books[::-1])
    return recommended_books


get_recommends("The Queen of the Damned (Vampire Chronicles (Paperback))")

['The Queen of the Damned (Vampire Chronicles (Paperback))',
 [['Catch 22', np.float32(0.7939835)],
  ['The Witching Hour (Lives of the Mayfair Witches)', np.float32(0.74486566)],
  ['Interview with the Vampire', np.float32(0.73450685)],
  ['The Tale of the Body Thief (Vampire Chronicles (Paperback))',
   np.float32(0.53763384)],
  ['The Vampire Lestat (Vampire Chronicles, Book II)',
   np.float32(0.51784116)]]]

In [15]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)


def test_book_recommendation():
    test_pass = True
    recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
    if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
        test_pass = False
    recommended_books = [
        "I'll Be Seeing You",
        "The Weight of Water",
        "The Surgeon",
        "I Know This Much Is True",
    ]
    recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
    for i in range(2):
        if recommends[1][i][0] not in recommended_books:
            test_pass = False
        if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
            test_pass = False
    if test_pass:
        print("You passed the challenge!")
    else:
        print("You haven't passed yet. Keep trying!")


test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", np.float32(0.8016211)], ['The Weight of Water', np.float32(0.77085835)], ['The Surgeon', np.float32(0.7699411)], ['I Know This Much Is True', np.float32(0.7677075)], ['The Lovely Bones: A Novel', np.float32(0.7234864)]]]
You passed the challenge!
